# Long responders and plateau diagnostics for the 69-claim CFR+ run

This notebook follows up the 12-hour `r6_s4_h4_hp2tfhq_ss` neural CFR+ run in two ways:

1. train fresh 60-minute action-conditioned fitted-return responders against snapshots nearest 200, 400, and 720 measured CFR+ minutes;
2. inspect the CFR+ training log for changes in iteration throughput, action coverage, importance-weight quality, records generated, replay retention, and compute allocation around the apparent 400-minute plateau.

The responder sweep is restartable at snapshot granularity. Completed targets are skipped; interrupted attempts are preserved and restarted in a new attempt directory.

In [ ]:
import gc
import json
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'liars_poker').is_dir():
    raise RuntimeError('Could not locate the liars_poker repository root.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.serialization import load_policy
from liars_poker.training.br_runs import run_best_responder

assert torch.cuda.is_available(), 'The long responder runs require CUDA.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('repository:', REPO_ROOT)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
spec = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)
assert len(rules_for_spec(spec).claims) == 69

# Set explicitly if automatic discovery chooses the wrong run.
REFERENCE_RUN_DIR = None
TARGET_CFR_MINUTES = (200, 400, 720)

BR_MINUTES = 60
BR_SEED = 7
BR_EVALUATE_EVERY_MINUTES = 5
BR_EVAL_EPISODES_PER_ROLE = 200_000
BR_EPISODES_PER_ROLE = 4096
BR_ROLLOUT_BATCH_SIZE = 1024
BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'device': device,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'learning_rate': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
}
print('spec:', spec.to_short_str())
print('total measured responder-training hours:', len(TARGET_CFR_MINUTES) * BR_MINUTES / 60)

## Locate the long CFR+ run and inspect its saved configuration

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

def matching_runs():
    root = REPO_ROOT / 'artifacts' / 'cfr_plus_approx_reference_runs'
    wanted = json.loads(spec.to_json())
    rows = []
    if not root.exists():
        return rows
    for run_dir in root.iterdir():
        manifest_path = run_dir / 'manifest.json'
        snapshot_root = run_dir / 'evaluations' / 'policy_snapshots' / 'snapshots'
        if not run_dir.is_dir() or not manifest_path.exists() or not snapshot_root.exists():
            continue
        manifest = read_json(manifest_path)
        raw_spec = manifest.get('spec')
        try:
            run_spec = json.loads(raw_spec) if isinstance(raw_spec, str) else raw_spec
        except (TypeError, json.JSONDecodeError):
            continue
        if run_spec != wanted:
            continue
        rows.append({
            'run directory': run_dir,
            'snapshots': sum(path.is_dir() for path in snapshot_root.iterdir()),
            'training min': float(manifest.get('measured_training_s', 0.0)) / 60.0,
            'iterations': int(manifest.get('iterations', 0)),
            'status': manifest.get('status'),
        })
    return sorted(rows, key=lambda row: (row['training min'], row['snapshots'], str(row['run directory'])))

candidates = matching_runs()
if candidates:
    display(pd.DataFrame(candidates).style.format({'training min': '{:.2f}'}))
if REFERENCE_RUN_DIR is None:
    if not candidates:
        raise FileNotFoundError('No matching long CFR+ run found. Set REFERENCE_RUN_DIR explicitly.')
    REFERENCE_RUN_DIR = Path(candidates[-1]['run directory'])
else:
    REFERENCE_RUN_DIR = Path(REFERENCE_RUN_DIR)

MANIFEST_PATH = REFERENCE_RUN_DIR / 'manifest.json'
TRAINING_PATH = REFERENCE_RUN_DIR / 'training.jsonl'
SNAPSHOT_ROOT = REFERENCE_RUN_DIR / 'evaluations' / 'policy_snapshots' / 'snapshots'
OLD_BR_ROOT = REFERENCE_RUN_DIR / 'approx_br_every_4th_20m'
OUTPUT_ROOT = REFERENCE_RUN_DIR / 'approx_br_selected_60m'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

manifest = read_json(MANIFEST_PATH)
trainer_kwargs = manifest.get('trainer_kwargs', {})
run_kwargs = manifest.get('run_kwargs', {})
configuration = pd.Series({
    'measured training min': float(manifest.get('measured_training_s', 0.0)) / 60.0,
    'iterations': int(manifest.get('iterations', 0)),
    'traversals per player per iteration': run_kwargs.get('traversals_per_player'),
    'total initial deals per iteration': (
        2 * int(run_kwargs.get('traversals_per_player', 0))
    ),
    'traversal batch size': trainer_kwargs.get('traversal_batch_size'),
    'sampled claims per traverser node': trainer_kwargs.get('traverser_action_sample_count'),
    'sample fraction': trainer_kwargs.get('traverser_action_sample_fraction'),
    'fully expand first traverser decision': trainer_kwargs.get('traverser_action_full_first'),
    'sampling baseline': trainer_kwargs.get('traverser_action_baseline'),
    'regret fit steps': trainer_kwargs.get('regret_train_steps'),
    'strategy fit steps': trainer_kwargs.get('strategy_train_steps'),
    'fit batch size': trainer_kwargs.get('batch_size'),
    'strategy buffer capacity per player': trainer_kwargs.get('strategy_buffer_capacity'),
})
print('reference run:', REFERENCE_RUN_DIR)
display(configuration.to_frame('value'))

## Select snapshots nearest 200, 400, and 720 minutes

In [ ]:
snapshot_rows = []
for policy_dir in SNAPSHOT_ROOT.iterdir():
    result_path = policy_dir / 'result.json'
    if not policy_dir.is_dir() or not (policy_dir / 'metadata.json').exists() or not result_path.exists():
        continue
    result = read_json(result_path)
    snapshot_rows.append({
        'policy_dir': policy_dir,
        'CFR+ training min': float(result.get('measured_training_s', 0.0)) / 60.0,
        'iteration': int(result.get('iteration', 0)),
    })

snapshots = pd.DataFrame(snapshot_rows).sort_values(['CFR+ training min', 'iteration']).reset_index(drop=True)
selected_rows = []
used_indices = set()
for target_min in TARGET_CFR_MINUTES:
    distances = (snapshots['CFR+ training min'] - target_min).abs()
    for idx in distances.sort_values().index:
        if int(idx) not in used_indices:
            used_indices.add(int(idx))
            row = snapshots.loc[idx].to_dict()
            row['target min'] = float(target_min)
            selected_rows.append(row)
            break
selected = pd.DataFrame(selected_rows).sort_values('target min').reset_index(drop=True)
_, loaded_spec = load_policy(str(selected.iloc[0]['policy_dir']))
assert loaded_spec == spec
selected.assign(policy_dir=selected['policy_dir'].astype(str)).to_csv(OUTPUT_ROOT / 'selected_snapshots.csv', index=False)
display(selected[['target min', 'CFR+ training min', 'iteration', 'policy_dir']].style.format({'target min': '{:.0f}', 'CFR+ training min': '{:.2f}'}))

## CFR+ training diagnostics

The phase summaries compare the early run, the approach to the apparent optimum, and the post-400-minute plateau. Importance-weight ESS near one means weights are relatively even; low ESS or rapidly growing maximum weights means sampled trajectories are being dominated by a small number of heavily corrected records.

In [ ]:
raw_training = read_jsonl(TRAINING_PATH)
if not raw_training:
    raise RuntimeError('training.jsonl is empty or missing.')
training = pd.DataFrame(raw_training).sort_values('measured_training_s').reset_index(drop=True)
timing = pd.json_normalize(training['timing']).add_prefix('timing.')
sampling = pd.json_normalize(training['action_sampling']).add_prefix('sampling.')
diag = pd.concat([training.drop(columns=['timing', 'action_sampling']), timing, sampling], axis=1)
diag['training min'] = diag['measured_training_s'] / 60.0
diag['regret records'] = diag['new_regret_records'].apply(lambda values: float(np.sum(values)))
diag['strategy records'] = diag['new_strategy_records'].apply(lambda values: float(np.sum(values)))
diag['regret loss mean'] = diag['regret_loss'].apply(lambda values: float(np.nanmean(values)))
diag['strategy loss mean'] = diag['strategy_loss'].apply(lambda values: float(np.nanmean(values)))
diag['strategy retained fraction'] = [
    np.sum(size) / max(1.0, np.sum(seen))
    for size, seen in zip(diag['strategy_buffer_sizes'], diag['strategy_records_seen'])
]
window = min(250, max(20, len(diag) // 30))
delta_iter = diag['iteration'].diff(window)
delta_min = diag['training min'].diff(window)
diag['rolling iterations / min'] = delta_iter / delta_min
diag['phase'] = pd.cut(
    diag['training min'],
    bins=[-np.inf, 200, 400, np.inf],
    labels=['0–200 min', '200–400 min', '400+ min'],
)

phase_summary = diag.groupby('phase', observed=True).agg(
    iterations=('iteration', 'count'),
    start_iteration=('iteration', 'min'),
    end_iteration=('iteration', 'max'),
    mean_iteration_s=('iteration_s', 'mean'),
    mean_traversal_s=('timing.traversal_s', 'mean'),
    mean_regret_fit_s=('timing.regret_training_s', 'mean'),
    mean_strategy_fit_s=('timing.strategy_training_s', 'mean'),
    mean_claim_edge_fraction=('sampling.claim_edge_fraction', 'mean'),
    mean_ESS_fraction=('sampling.regret_weight_ess_fraction', 'mean'),
    maximum_regret_weight=('sampling.max_regret_weight', 'max'),
    mean_regret_records=('regret records', 'mean'),
    mean_strategy_records=('strategy records', 'mean'),
    final_strategy_retained_fraction=('strategy retained fraction', 'last'),
    mean_regret_loss=('regret loss mean', 'mean'),
    mean_strategy_loss=('strategy loss mean', 'mean'),
)
display(phase_summary.style.format(precision=5))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True)
x = diag['training min']

axes[0, 0].plot(x, diag['iteration_s'], alpha=0.25)
axes[0, 0].plot(x, diag['iteration_s'].rolling(window, min_periods=1).mean(), linewidth=2)
axes[0, 0].set(title='Iteration cost', ylabel='Seconds')

axes[0, 1].plot(x, diag['rolling iterations / min'])
axes[0, 1].set(title=f'Iteration throughput ({window}-iteration window)', ylabel='Iterations / min')

axes[0, 2].plot(x, diag['timing.traversal_s'], label='traversal', alpha=0.7)
axes[0, 2].plot(x, diag['timing.regret_training_s'], label='regret fit', alpha=0.7)
axes[0, 2].plot(x, diag['timing.strategy_training_s'], label='strategy fit', alpha=0.7)
axes[0, 2].set(title='Compute allocation', ylabel='Seconds')
axes[0, 2].legend()

axes[1, 0].plot(x, diag['sampling.claim_edge_fraction'], label='claim-edge fraction')
axes[1, 0].plot(x, diag['sampling.regret_weight_ess_fraction'], label='weight ESS fraction')
axes[1, 0].set(title='Action-sampling coverage and weight quality', ylabel='Fraction')
axes[1, 0].legend()

axes[1, 1].semilogy(x, diag['sampling.max_regret_weight'].clip(lower=1.0))
axes[1, 1].set(title='Largest inverse-path regret weight', ylabel='Weight')

axes[1, 2].plot(x, diag['strategy retained fraction'], label='strategy retained fraction')
axes[1, 2].set(title='Average-strategy reservoir retention', ylabel='Fraction')

for ax in axes.flat:
    ax.axvline(200, color='grey', linestyle='--', alpha=0.5)
    ax.axvline(400, color='grey', linestyle='--', alpha=0.5)
    ax.grid(alpha=0.25)
for ax in axes[1]:
    ax.set_xlabel('Measured CFR+ training minutes')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True)
axes[0].plot(x, diag['regret records'], alpha=0.45, label='regret records')
axes[0].plot(x, diag['strategy records'], alpha=0.45, label='strategy records')
axes[0].set(title='Records generated per iteration', ylabel='Records')
axes[0].legend()

axes[1].plot(x, diag['regret loss mean'], alpha=0.45)
axes[1].plot(x, diag['regret loss mean'].rolling(window, min_periods=1).mean(), linewidth=2)
axes[1].set(title='Regret fitting loss', ylabel='Loss')

axes[2].plot(x, diag['strategy loss mean'], alpha=0.45)
axes[2].plot(x, diag['strategy loss mean'].rolling(window, min_periods=1).mean(), linewidth=2)
axes[2].set(title='Average-network fitting loss', ylabel='Loss')

for ax in axes:
    ax.axvline(200, color='grey', linestyle='--', alpha=0.5)
    ax.axvline(400, color='grey', linestyle='--', alpha=0.5)
    ax.set_xlabel('Measured CFR+ training minutes')
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Run fresh 60-minute responders

Set `RUN_LONG_RESPONDERS = True`. The cell skips completed targets. If interrupted during one target, rerunning starts a new attempt only for that target.

In [ ]:
def attempt_directories(target_root):
    return sorted(path for path in target_root.glob('attempt_*') if path.is_dir())

def completed_attempt(target_root):
    for attempt in reversed(attempt_directories(target_root)):
        if (attempt / 'metrics.json').exists() and (attempt / 'evaluations.jsonl').exists():
            return attempt
    return None

def next_attempt_directory(target_root):
    existing = attempt_directories(target_root)
    number = 1 + max((int(path.name.split('_')[-1]) for path in existing), default=0)
    return target_root / f'attempt_{number:02d}'

def target_output_root(row):
    return OUTPUT_ROOT / f"target_{int(row['target min']):04d}m"

free_gib = shutil.disk_usage(REFERENCE_RUN_DIR).free / 2**30
print('free disk GiB:', round(free_gib, 3))
if free_gib < 0.5:
    raise RuntimeError('Less than 0.5 GiB free. Clean the disk before starting.')

RUN_LONG_RESPONDERS = False
if RUN_LONG_RESPONDERS:
    for position, row in selected.iterrows():
        target_root = target_output_root(row)
        target_root.mkdir(parents=True, exist_ok=True)
        done = completed_attempt(target_root)
        if done is not None:
            print(f"[{position + 1}/3] target {row['target min']:.0f}m: skipping {done.name}")
            continue

        attempt_dir = next_attempt_directory(target_root)
        kwargs = dict(BR_TRAINER_KWARGS)
        kwargs['seed'] = BR_SEED
        print(
            f"\n[{position + 1}/3] target {row['target min']:.0f}m "
            f"(actual {row['CFR+ training min']:.2f}m) -> {attempt_dir.name}"
        )
        result = run_best_responder(
            row['policy_dir'],
            method='action_conditioned_fitted_return',
            minutes=BR_MINUTES,
            trainer_kwargs=kwargs,
            episodes_per_role=BR_EPISODES_PER_ROLE,
            rollout_batch_size=BR_ROLLOUT_BATCH_SIZE,
            evaluate_every_minutes=BR_EVALUATE_EVERY_MINUTES,
            eval_episodes_per_role=BR_EVAL_EPISODES_PER_ROLE,
            exact_evaluation=False,
            run_dir=attempt_dir,
            debug=True,
        )
        del result
        gc.collect()
        torch.cuda.empty_cache()
    print('\nLong responder sweep complete.')

## Compare 60-minute responder curves with the existing 20-minute sweep

In [ ]:
def monotone_curve(path):
    frame = pd.DataFrame(read_jsonl(path))
    if frame.empty:
        return frame
    frame = frame.sort_values('measured_training_s').copy()
    frame['best p_first'] = frame['p_first'].cummax()
    frame['best p_second'] = frame['p_second'].cummax()
    frame['best p_first LCB'] = frame['p_first_lcb'].cummax()
    frame['best p_second LCB'] = frame['p_second_lcb'].cummax()
    frame['best discovered estimate'] = frame['best p_first'] + frame['best p_second'] - 1.0
    frame['conservative lower bound'] = frame['best p_first LCB'] + frame['best p_second LCB'] - 1.0
    return frame

long_frames = []
summary_rows = []
for _, row in selected.iterrows():
    attempt = completed_attempt(target_output_root(row))
    if attempt is None:
        continue
    frame = monotone_curve(attempt / 'evaluations.jsonl')
    if frame.empty:
        continue
    frame['target min'] = float(row['target min'])
    frame['CFR+ training min'] = float(row['CFR+ training min'])
    long_frames.append(frame)
    final = frame.iloc[-1]
    summary_rows.append({
        'target min': float(row['target min']),
        'CFR+ training min': float(row['CFR+ training min']),
        'snapshot iteration': int(row['iteration']),
        'responder training min': float(final['measured_training_s']) / 60.0,
        'best p_first': float(final['best p_first']),
        'best p_second': float(final['best p_second']),
        'best discovered estimate': float(final['best discovered estimate']),
        'conservative lower bound': float(final['conservative lower bound']),
        'attempt directory': str(attempt),
    })

if not long_frames:
    raise RuntimeError('No completed 60-minute responders found yet.')
long_curves = pd.concat(long_frames, ignore_index=True)
long_summary = pd.DataFrame(summary_rows).sort_values('target min')
long_curves.to_csv(OUTPUT_ROOT / 'combined_60m_curves.csv', index=False)
long_summary.to_csv(OUTPUT_ROOT / '60m_summary.csv', index=False)
display(long_summary.style.format(precision=6))

old_summary_path = OLD_BR_ROOT / 'snapshot_summary.csv'
old_summary = pd.read_csv(old_summary_path) if old_summary_path.exists() else pd.DataFrame()
if not old_summary.empty:
    display(old_summary.tail().style.format(precision=6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2))

if not old_summary.empty:
    axes[0].plot(
        old_summary['CFR+ training min'],
        old_summary['best discovered estimate'],
        color='0.7',
        marker='o',
        label='20-minute responder sweep',
    )
axes[0].plot(
    long_summary['CFR+ training min'],
    long_summary['best discovered estimate'],
    marker='o',
    linewidth=2.5,
    label='60-minute responders',
)
axes[0].plot(
    long_summary['CFR+ training min'],
    long_summary['conservative lower bound'],
    marker='o',
    linewidth=2.0,
    label='60-minute conservative lower bound',
)
axes[0].set(
    title='Does the post-400-minute plateau survive stronger BRs?',
    xlabel='Measured CFR+ training minutes',
    ylabel='Discovered exploitability',
)
axes[0].grid(alpha=0.3)
axes[0].legend()

for target_min, frame in long_curves.groupby('target min', sort=True):
    axes[1].plot(
        frame['measured_training_s'] / 60.0,
        frame['best discovered estimate'],
        marker='o',
        label=f'CFR+ target {target_min:.0f}m',
    )
axes[1].set(
    title='Long responder compute curves',
    xlabel='Measured responder-training minutes',
    ylabel='Best discovered exploitability',
)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## Reading the outcome

- If all three 60-minute curves approach the same value, later CFR+ snapshots became harder to exploit quickly but did not become materially less exploitable.
- If the 400- or 720-minute policies remain lower after the curves flatten, some genuine improvement was hidden by the 20-minute evaluator budget.
- Falling ESS, rising maximum weights, or declining records per initial deal implicate action-sampling variance.
- Stable sampling diagnostics combined with flat policy quality implicate the neural CFR+ target/fitting dynamics rather than a gradual operational slowdown.

This run used 256 traversals per player per iteration: 512 initial deals total. For the next training experiment, the cleanest first comparison is not simply more fit steps. It is increased traversal coverage and early-depth expansion: for example 512 or 1024 traversals per player, while fully expanding the first traverser decision before applying the cap. Fully expanding the first two or more traverser decisions needs a small code generalization and should be profiled because frontier growth can be substantial.